## Step 1 : install and load the libs

In [ ]:
#install the libs

!pip install -q transformers peft bitsandbytes trl datasets accelerate

In [ ]:
!pip install bitsandbytes==0.46.1 --quiet


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
#import the required libs

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [ ]:
# adding the hf token and login

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
login(token=secret.get_secret("HF_TOKEN_LLAMA"))
print("HF login done")

## step 2 : Load the data from hf and write the prompt format for training

In [ ]:
# loading the dataset from hf

dataset = load_dataset("spider")
print(dataset)

In [7]:
print(dataset["train"][0])

{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?']}


In [8]:
# engineered prompt for training and inference

def format_prompt(example):
    return {
        "text": f"""### Task
Generate a SQL query based on the question and schema below.

### Schema
{example['db_id']}

### Question
{example['question']}

### SQL
{example['query']}"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [9]:
print(dataset.keys())

dict_keys(['train', 'validation'])


In [10]:
print(dataset["train"][0])

{'db_id': 'department_management', 'query': 'SELECT count(*) FROM head WHERE age  >  56', 'question': 'How many heads of the departments are older than 56 ?', 'query_toks': ['SELECT', 'count', '(', '*', ')', 'FROM', 'head', 'WHERE', 'age', '>', '56'], 'query_toks_no_value': ['select', 'count', '(', '*', ')', 'from', 'head', 'where', 'age', '>', 'value'], 'question_toks': ['How', 'many', 'heads', 'of', 'the', 'departments', 'are', 'older', 'than', '56', '?'], 'text': '### Task\nGenerate a SQL query based on the question and schema below.\n\n### Schema\ndepartment_management\n\n### Question\nHow many heads of the departments are older than 56 ?\n\n### SQL\nSELECT count(*) FROM head WHERE age  >  56'}


In [11]:
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print("Question:", dataset['train'][i]['question'])
    print("SQL:", dataset['train'][i]['query'])
    print("DB:", dataset['train'][i]['db_id'])
    print()

--- Sample 1 ---
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56
DB: department_management

--- Sample 2 ---
Question: List the name, born state and age of the heads of departments ordered by age.
SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age
DB: department_management

--- Sample 3 ---
Question: List the creation year, name and budget of each department.
SQL: SELECT creation ,  name ,  budget_in_billions FROM department
DB: department_management



In [12]:
dataset = dataset.map(format_prompt)
print(dataset['train'][0]['text'])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

### Task
Generate a SQL query based on the question and schema below.

### Schema
department_management

### Question
How many heads of the departments are older than 56 ?

### SQL
SELECT count(*) FROM head WHERE age  >  56


## step 3 : Load the model and tokenizer from the  and quantized it


In [13]:
# configure the quantization

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [14]:
# adding this to clear memory as loading a 16gb model in the follwoing cell

import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [15]:
model_name = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True

)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

## step 4 : Lora Adapter Configuration(PEFT)

In [25]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#prepare model for kbit trainig, 1)freeze base layers 2)gradeint checkpoint
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r = 16,               #4096x16 16x4096
    lora_alpha = 32,      # standard twice as r
    target_modules = [    # attention layers are targeted only since most of the relations building is learned here i.e between language and sql
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout = 0.1,   #regularization techniqe used for preventing overfitting
    bias = "none",         # just not worth training
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)   # just attaching the layers A and B to the oringinal linear layer but it has no effect intially as its intialized with 0

model.print_trainable_parameters() 

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


## Step 5a: SFTTrainer Setup
its a trl library built on hugging face, controls the entire training loop, forward pass,loss computation , gradient accumalation and checkpointing

In [17]:
#test code 
import trl
print(trl.__version__)

1.2.0


In [18]:
# install mlflow

!pip install mlflow --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.2 MB/s eta 0:00:00


In [30]:
# mlflow setup 
import mlflow
import os

os.environ["MLFLOW_TRACKING_URI"] = "/kaggle/working/mlruns"
mlflow.set_experiment("llama3-text2sql")
print("MLflow ready")

MLflow ready


In [31]:
from trl import SFTTrainer, SFTConfig

# trainig config
training_args = SFTConfig(
    output_dir = "Awan8754/llama_nl_to_sql",
    num_train_epochs = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=13,
    bf16=True,
    logging_steps=10, #print loss after every 10 steps
    eval_strategy="steps", # one step is after how many samples the weight get updates so batch_size x gradient_accumalation steps
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataset_text_field="text",
    report_to="mlflow",
    packing=False,
    push_to_hub=True,
    hub_strategy="checkpoint",
)

#intialize the trainer
trainer = SFTTrainer(
    model = model,
    args = training_args,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    processing_class = tokenizer

)

## Step 5b : MLflow integration

In [32]:
import mlflow
from transformers import TrainerCallback

# make a callback function so it can bridge the trainer logs with mlflow
class MLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        # `logs` is a dict like {"loss": 0.43, "eval_loss": 0.38, "epoch": 1.2}
        # `state.global_step` is the current step number
        if logs:
            for key, value in logs.items():
                if isinstance(value, (int, float)):  # skip non-numeric
                    mlflow.log_metric(key, value, step=state.global_step)

# --- MLflow setup ---
mlflow.set_tracking_uri("file:///kaggle/working/mlruns")  # save locally

# --- Wrap training in a run ---
with mlflow.start_run(run_name="llama3.1-8b-spider-v2"):

    # Log all your hyperparams once
    mlflow.log_params({
        "model_name": "meta-llama/Meta-Llama-3.1-8B",
        "learning_rate": 1e-4,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "num_epochs": 1,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
        "max_seq_length": 512,
    })

    # Attach MLflow callback to trainer
    trainer.add_callback(MLflowCallback())

    # Train
    trainer.train(resume_from_checkpoint="Awan8754/llama_nl_to_sql/checkpoint-100")

    # Save adapter + tokenizer
    trainer.model.save_pretrained("./final_adapter")
    tokenizer.save_pretrained("./final_adapter")

    # Log the saved folder as artifact
    mlflow.log_artifacts("./final_adapter", artifact_path="final_adapter")

print("Training done. MLflow run saved to /kaggle/working/mlruns")

Step,Training Loss,Validation Loss
200,0.752092,0.903771
300,0.659443,0.915167
400,0.608407,0.969366


Training done. MLflow run saved to /kaggle/working/mlruns


In [24]:
#only for the mlflow part,

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import shutil

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN_LLAMA")
api = HfApi()

shutil.make_archive("mlruns_backup", "zip", "/kaggle/working/mlruns")
api.upload_file(
    path_or_fileobj="mlruns_backup.zip",
    path_in_repo="mlruns_backup.zip",
    repo_id="Awan8754/llama_nl_to_sql",
    repo_type="model",
    token=hf_token
)
print("MLflow logs saved to HF.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

MLflow logs saved to HF.


In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import shutil

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN_LLAMA")
login(token=hf_token)

api = HfApi()

# push adapter
api.upload_folder(
    folder_path="./final_adapter",
    repo_id="Awan8754/llama3.1-text2sql-adapter",
    repo_type="model",
    token=hf_token
)

# zip and push mlruns
shutil.make_archive("mlruns_backup", "zip", "/kaggle/working/mlruns")
api.upload_file(
    path_or_fileobj="mlruns_backup.zip",
    path_in_repo="mlruns_backup.zip",
    repo_id="Awan8754/llama_nl_to_sql",
    repo_type="model",
    token=hf_token
)

print("Everything saved to HF. Sleep well.")

## Step 6 : now lets test the trained adapter on the exact matching
 as spider dataset on hf doenst got the test-data

In [1]:
#install required libs
!pip install -q transformers peft bitsandbytes datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.3 MB/s eta 0:00:00:00:0100:01


In [2]:
# # adding the hf token and login

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
login(token=secret.get_secret("HF_TOKEN_LLAMA"))
print("HF login done")

HF login done


In [3]:
# load the model and imports

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
import torch

# load the model in nf4 and then expand to bfloat16 why ?
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# model and my adapter path
base_model = "meta-llama/Llama-3.1-8B"
adapter = "Awan8754/llama_nl_to_sql"

# pull the tokenizer and the padding tokens
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map={"": 0}   # use to put all the layers on gpu0 sometimes if u do it "auto" it messes with QLORA
)
model = PeftModel.from_pretrained(model, adapter)
model.eval()    # did this to switch it to evaluation mode so switching off batchnorm and dropout
print("Model loaded")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/983 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

Model loaded


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.l

In [4]:
# now we gotta load the dataset and then we gotta write prompt for evaluation coz training prompt has answer in it
dataset = load_dataset("spider")

def format_eval_prompt(example):
    return f"""### Task
Generate a SQL query based on the question and schema below.
### Schema
{example['db_id']}
### Question
{example['question']}
### SQL
"""

val_data = dataset["validation"]


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

In [5]:
print(format_eval_prompt(val_data[0]))


### Task
Generate a SQL query based on the question and schema below.
### Schema
concert_singer
### Question
How many singers do we have?
### SQL



In [6]:
# generation and evalaution is happeining over here, 

def generate_sql(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,        # greedy decoding for SQL
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # extract only what's after ### SQL
    sql = generated.split("### SQL")[-1].strip()
    return sql

correct = 0
val_subset = val_data.select(range(50))
total = len(val_subset)

for i, example in enumerate(val_subset):
    prompt = format_eval_prompt(example)
    predicted_sql = generate_sql(prompt)
    gold_sql = example["query"].strip()
    
    if predicted_sql.lower() == gold_sql.lower():  # case insensitive match
        correct += 1
    
    if i % 100 == 0:
        print(f"Progress: {i}/{total} — running accuracy: {correct/(i+1):.3f}")

print(f"\nFinal Exact Match Accuracy: {correct/total:.3f} ({correct}/{total})")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Progress: 0/50 — running accuracy: 0.000

Final Exact Match Accuracy: 0.000 (0/50)


In [7]:
example = val_data[0]
prompt = format_eval_prompt(example)
predicted = generate_sql(prompt)
gold = example["query"].strip()

print("=== PROMPT ===")
print(prompt)
print("=== PREDICTED ===")
print(predicted)
print("=== GOLD ===")
print(gold)

=== PROMPT ===
### Task
Generate a SQL query based on the question and schema below.
### Schema
concert_singer
### Question
How many singers do we have?
### SQL

=== PREDICTED ===
SELECT COUNT(*) FROM concert_singer;
=== GOLD ===
SELECT count(*) FROM singer


In [13]:
for i, example in enumerate(val_data.select(range(3))):
    prompt = format_eval_prompt(example)
    predicted = generate_sql(prompt)
    gold = example["query"].strip()
    print(f"=== Example {i} ===")
    print(f"PREDICTED: {predicted}")
    print(f"GOLD:      {gold}")
    print()

=== Example 0 ===
PREDICTED: SELECT COUNT(*) FROM concert_singer;
GOLD:      SELECT count(*) FROM singer

=== Example 1 ===
PREDICTED: SELECT COUNT(*) FROM concert_singer;
GOLD:      SELECT count(*) FROM singer

=== Example 2 ===
PREDICTED: ```sql
SELECT name, country, age FROM concert_singer ORDER BY age DESC;
```
GOLD:      SELECT name ,  country ,  age FROM singer ORDER BY age DESC



### Side quest

In [8]:
from datasets import load_dataset
spider_tables = load_dataset("spider", split="train").features
# actually do this:
print(dataset["validation"][0].keys())

dict_keys(['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'])


In [9]:
import json
import os

# Spider tables.json path on Kaggle
tables_path = "/root/.cache/huggingface/datasets/spider/main/tables.json"
print(os.path.exists(tables_path))

False


In [10]:
import subprocess
result = subprocess.run(["find", "/root/.cache", "-name", "tables.json"], capture_output=True, text=True)
print(result.stdout)

In [11]:
import requests
import json

url = "https://raw.githubusercontent.com/taoyds/spider/master/tables.json"
response = requests.get(url)
tables = json.loads(response.text)
print(f"Loaded {len(tables)} databases")
print(tables[0].keys())

JSONDecodeError: Extra data: line 1 column 4 (char 3)

In [12]:
print(response.status_code)
print(response.text[:200])

404
404: Not Found
